### Embeddings, Static, Contextual, and Applied

Static word vectors (Word2Vec, FastText) assign one vector per word regardless of context. Contextual embeddings (BERT) fix that. The last section shows pretrained embeddings feeding a downstream head (QA, sentiment).

#### 1. Static Embeddings - Word2Vec

##### Skip-gram

- Goal: predict context words from a target (center) word.
- Example: "He sat by the river bank", center="bank", window=2 → context=["the", "river"].
- Training pair: input "bank" → outputs "the", "river".
- Best for: rare words.

##### CBOW (Continuous Bag of Words)

- Goal: predict the target (center) word from its context words.
- Example: "He sat by the river bank", context=["the", "river"] → predict "bank".
- Training pair: input ["the", "river"] → output "bank".
- Best for: faster training, frequent words.

In [1]:
# gensim: library for word vector / topic models
from gensim.models import Word2Vec

# Basic tokenizer: splits sentences into lowercase words
def simple_tokenizer(text):
    # Lowercase and split on spaces (very basic, for demo only)
    return text.lower().replace('.', '').split()

# Example sentences with the word 'bank' in different contexts
raw_sentences = [
    "He sat by the river bank.",
    "She deposited money in the bank.",
    "The bank was closed on Sunday."
]

# Tokenize sentences
tokenized_sentences = [simple_tokenizer(sent) for sent in raw_sentences]
# [He, Sat, by, the]

# Train Word2Vec model
# vector_size: dimension of the embedding vectors (higher = more expressive, but needs more data)
# min_count: ignore words with total frequency lower than this (set to 1 to include all words in this tiny corpus)
# window: maximum distance between the current and predicted word within a sentence (context window size)
w2v_model = Word2Vec(tokenized_sentences, vector_size=50, min_count=1, window=3)
# This is Skip-gram (sg=1) by default and sg=0 for CBOW

# Get embedding for 'bank'
# The vector for 'bank' will be the same regardless of which sentence/context it appears in
print("Word2Vec vector for 'bank':\n", w2v_model.wv['bank'])
print("Word2Vec : Bank's Word Vector Shape", w2v_model.wv['bank'].shape)

words = list(w2v_model.wv.index_to_key)  # List all words in the vocabulary
similar = w2v_model.wv.most_similar('bank')  # Find words most similar to 'bank'

print("Similarity between 'bank' and 'river':", w2v_model.wv.similarity('bank', 'river'))
print("Similarity between 'bank' and 'money':", w2v_model.wv.similarity('bank', 'money'))

# NOTE: sliding window learns similar contexts -> similar vectors.
# 'bank' gets one vector averaging all its contexts (river + money) -- static embeddings
# can't distinguish word senses (polysemy).

Word2Vec vector for 'bank':
 [-1.0724545e-03  4.7286271e-04  1.0206699e-02  1.8018546e-02
 -1.8605899e-02 -1.4233618e-02  1.2917745e-02  1.7945977e-02
 -1.0030856e-02 -7.5267432e-03  1.4761009e-02 -3.0669428e-03
 -9.0732267e-03  1.3108104e-02 -9.7203208e-03 -3.6320353e-03
  5.7531595e-03  1.9837476e-03 -1.6570430e-02 -1.8897636e-02
  1.4623532e-02  1.0140524e-02  1.3515387e-02  1.5257311e-03
  1.2701781e-02 -6.8107317e-03 -1.8928028e-03  1.1537147e-02
 -1.5043275e-02 -7.8722071e-03 -1.5023164e-02 -1.8600845e-03
  1.9076237e-02 -1.4638334e-02 -4.6675373e-03 -3.8754821e-03
  1.6154874e-02 -1.1861792e-02  9.0324880e-05 -9.5074680e-03
 -1.9207101e-02  1.0014586e-02 -1.7519170e-02 -8.7836506e-03
 -7.0199967e-05 -5.9236289e-04 -1.5322480e-02  1.9229487e-02
  9.9641159e-03  1.8466286e-02]
Word2Vec : Bank's Word Vector Shape (50,)
Similarity between 'bank' and 'river': -0.012591083
Similarity between 'bank' and 'money': 0.13204393


#### 1.1 Static Embeddings - FastText

In [2]:
from gensim.models import FastText

# Train FastText model on your tokenized sentences
ft_model = FastText(tokenized_sentences, vector_size=10, min_count=1, window=3)

# Get embedding for a known word
# Each word is broken into character n-grams (e.g., "bank" → <ba, ban, ank, nk>, etc.).
# The word vector is the sum (or average) of its n-gram vectors.
# Training is similar to Word2Vec (CBOW or Skip-gram), but on subwords.
print("FastText vector for 'bank':\n", ft_model.wv['bank'])

# FastText can handle OOV (out-of-vocabulary) words using subword information
print("FastText vector for OOV word 'banking':\n", ft_model.wv['banking'])

# Compare with a nonsense word (still gets a vector!)
print("FastText vector for OOV word 'bankzzz':\n", ft_model.wv['bankzzz'])

# Demo: Similarity between 'bank' and 'banking'
print("Similarity between 'bank' and 'banking':", ft_model.wv.similarity('bank', 'banking'))

# NOTE:
# - FastText is especially useful when you expect to encounter new words,
# - rare words, or work with morphologically rich languages.

FastText vector for 'bank':
 [-0.02681367 -0.00070708  0.01805622  0.00670038 -0.00706677 -0.01905808
  0.01269603  0.00208769  0.00945567 -0.02328101]
FastText vector for OOV word 'banking':
 [ 0.00135347 -0.00299125  0.00331595  0.01845975  0.0230606   0.00761079
  0.00130226  0.00841182  0.00700071 -0.01047717]
FastText vector for OOV word 'bankzzz':
 [-0.02014219  0.00406886 -0.00836108  0.01079835  0.01020753  0.00953312
  0.01484188 -0.01233316  0.0123117  -0.02921248]
Similarity between 'bank' and 'banking': 0.112472326


#### 2. Context-Aware Embeddings - BERT

- Word2Vec/GloVe assign one vector per word regardless of context, problematic for polysemy (e.g. "bank": river vs. financial), the limitation demonstrated above.
- BERT generates different embeddings for the same word depending on sentence context.

In [3]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT model and tokenizer
# The tokenizer splits sentences into tokens
# that BERT understands, including handling subwords.

# the embedding (hidden) dimension is 768
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
# model.config.hidden_size == 768, Can't change, pre-determined as pre-trained

/Users/monusingh/work-share/code-blogs-articles/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/199 [00:00<00:00, 4169.29it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|          | 1/199 [00:00<00:00, 995.33it/s, Materializing param=embeddings.LayerNorm.bias] 

Loading weights:   1%|          | 2/199 [00:00<00:00, 665.82it/s, Materializing param=embeddings.LayerNorm.weight]

...

Loading weights:  99%|█████████▉| 198/199 [00:00<00:00, 2146.20it/s, Materializing param=pooler.dense.bias]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2152.38it/s, Materializing param=pooler.dense.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2148.29it/s, Materializing param=pooler.dense.weight]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2142.85it/s, Materializing param=pooler.dense.weight]


BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Two sentences with "bank" in different contexts
sentences = [
    "He sat by the river bank.",
    "She deposited money in the bank."
]

def get_word_embedding(sentence, target_word):
    '''
    Tokenizes the sentence.
    Pass through BERT, get the last hidden state
    Find token corresponding to the target word which is "bank"
    '''

    # Tokenize and get input IDs
    inputs = bert_tokenizer(sentence, return_tensors='pt') # pt stands for PyTorch, ;tf for tensorfow
    # For BERT, the maximum input length is 512 tokens (including special tokens like [CLS] and [SEP]).
    # inputs = tokenizer(sentence, return_tensors='pt', max_length=512, truncation=True)

    with torch.no_grad(): # for inference, no gradient calc, hence faster & less memory
        outputs = bert_model(**inputs) # input is dict - {'input_ids': ..., 'attention_mask': ...}

    # Get the last hidden state (batch_size, seq_len, hidden_size)
    # i.e the output of final layer for each token in input
    last_hidden_state = outputs.last_hidden_state.squeeze(0) # removes firs dim, if size =1

    # Decode tokens to align with input words
    tokens = bert_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Find the index of the target word (may need to handle subwords)
    # We'll take the first occurrence for simplicity
    for i, token in enumerate(tokens):
        if target_word in token:
            return last_hidden_state[i].numpy(), tokens
    return None, tokens

# Get embeddings for "bank" in both sentences
vec1, tokens1 = get_word_embedding(sentences[0], "bank")
vec2, tokens2 = get_word_embedding(sentences[1], "bank")

print("Tokens in sentence 1:", tokens1)
print("Tokens in sentence 2:", tokens2)
print("Embedding for 'bank' in sentence 1 (river context):", vec1[:5])  # Show first 5 dims
print("Embedding for 'bank' in sentence 2 (money context):", vec2[:5])

Tokens in sentence 1: ['[CLS]', 'he', 'sat', 'by', 'the', 'river', 'bank', '.', '[SEP]']
Tokens in sentence 2: ['[CLS]', 'she', 'deposited', 'money', 'in', 'the', 'bank', '.', '[SEP]']
Embedding for 'bank' in sentence 1 (river context): [ 0.15994893 -0.3381438  -0.03246736 -0.08658531 -0.39891723]
Embedding for 'bank' in sentence 2 (money context): [ 0.3031029  -0.36687204 -0.35636535  0.1448594   1.0418973 ]


In [5]:
from numpy import dot
from numpy.linalg import norm

def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))

similarity = cosine_similarity(vec1, vec2)
print("Cosine similarity between 'bank' in different contexts:", similarity)
# the vectors are different, and their similarity will be less than 1

Cosine similarity between 'bank' in different contexts: 0.52787507


1. MLM (Masked Language Modeling)
- BERT's main pre-training task: some input words are randomly replaced with [MASK].
- Model predicts the original word using context from both sides, teaches word/context relationships.

2. NSP (Next Sentence Prediction)
- Given a sentence pair, predict whether the second logically follows the first (50% real, 50% random).
- Teaches sentence-level relationships, useful for QA and NLI.

3. Bidirectional
- BERT encodes each word using the entire sentence (both left and right context) at once.
- Unlike GPT/LSTM (left-to-right or right-to-left only), full context makes embeddings more powerful.

#### 3. Applying Pretrained Embeddings, QA and Sentiment Pipelines

Shows contextual embeddings from a pretrained model (DistilBERT) consumed by a downstream head to produce task outputs: extractive question-answering and sentiment classification.

In [6]:
from transformers import pipeline

# Load a pre-trained question-answering model, e.g., BERT fine-tuned on SQuAD
qa_pipeline = pipeline("question-answering", model="distilbert-base-uncased-distilled-squad")

context = """
The Amazon River is the largest river by discharge volume of water in the world,
and by some definitions, it is the longest. It flows through Peru,
Colombia, and Brazil before emptying into the Atlantic Ocean.
"""
question = "Which countries does the Amazon River flow through?"

result = qa_pipeline(question=question, context=context)

print(f"Question: {question}")
print(f"Answer: {result['answer']}")
print(f"Score: {result['score']:.2f}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/102 [00:00<00:00, 59074.70it/s, Materializing param=distilbert.embeddings.LayerNorm.bias]

Loading weights:   1%|          | 1/102 [00:00<00:00, 723.65it/s, Materializing param=distilbert.embeddings.LayerNorm.bias]  

Loading weights:   2%|▏         | 2/102 [00:00<00:00, 641.23it/s, Materializing param=distilbert.embeddings.LayerNorm.weight]

Loading weights:   2%|▏         | 2/102 [00:00<00:00, 537.77it/s, Materializing param=distilbert.embeddings.LayerNorm.weight]

...

Loading weights:  99%|█████████▉| 101/102 [00:00<00:00, 2115.93it/s, Materializing param=qa_outputs.bias]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2128.81it/s, Materializing param=qa_outputs.weight]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2120.61it/s, Materializing param=qa_outputs.weight]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2111.80it/s, Materializing param=qa_outputs.weight]

Question: Which countries does the Amazon River flow through?
Answer: Peru,
Colombia, and Brazil
Score: 0.99


In [7]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, pipeline

# --- Step 1: Load Pre-trained Tokenizer and Model ---
# The tokenizer is responsible for converting raw text into numerical IDs that the model understands.
# It also handles special tokens (like [CLS], [SEP]) and subword tokenization.
print("--- Step 1: Loading Tokenizer and Model ---")
distil_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# DistilBertForSequenceClassification is a DistilBERT model with a classification head
# (a linear layer) on top, specifically designed for tasks like sentiment analysis.
# This model has been pre-trained on a massive text corpus and then
# fine-tuned on a sentiment dataset (implicitly, by using a model often associated with sentiment).
# For a raw pre-trained model that hasn't been fine-tuned for classification yet,
# you would typically use `DistilBertModel` and add your own classification head.
# Here, we're simulating a common scenario where a pre-trained model *is* the sentiment analyzer.
# We're loading a general DistilBERT, then we'll show how its outputs are used for classification.
# For a truly 'ready-to-go' sentiment model, you'd load one specifically fine-tuned for it.
# Let's load the *base* model first to show the embedding process.
# We will later use a `pipeline` for an end-to-end solution.
model_base = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
print("Tokenizer and base model loaded.")

# --- Step 2: Prepare Input Text ---
# We'll use a sample sentence for sentiment analysis.
text = "This movie was absolutely fantastic! I loved every minute of it."
print(f"\n--- Step 2: Input Text ---")
print(f"Original Text: '{text}'")

# --- Step 3: Tokenize the Input Text ---
# The tokenizer converts the text into a sequence of numerical IDs (input_ids),
# and also generates an attention mask.
# `input_ids`: numerical representation of tokens.
# `attention_mask`: indicates which tokens are real words (1) and which are padding (0).
# `return_tensors='pt'` ensures the output is a PyTorch tensor.
print("\n--- Step 3: Tokenization ---")
inputs = distil_tokenizer(text, return_tensors='pt', padding=True, truncation=True)
print(f"Tokenized Input IDs: {inputs['input_ids']}")
print(f"Attention Mask: {inputs['attention_mask']}")
print(f"Decoded Tokens: {distil_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
# Note: [CLS] is a special token at the beginning for classification tasks.
# [SEP] is a special token to separate sentences. DistilBERT does not explicitly use [SEP] for single sentences,
# but it's common in BERT-like models for pair tasks.

# --- Step 4: Obtain Embeddings (and logits) from the Pre-trained Model ---
# When you pass the tokenized inputs to the DistilBertForSequenceClassification model,
# it first generates contextual embeddings for each token.
# Then, a classification head (a dense layer) on top of these embeddings
# produces logits (raw scores) for each possible class (e.g., positive, negative).
print("\n--- Step 4: Model Forward Pass & Logits ---")
# When `output_hidden_states=True` is passed, the model will also return
# the hidden states (which include the embeddings at different layers).
outputs = model_base(**inputs, output_hidden_states=True)

# The `logits` are the raw, unnormalized scores for each class.
# For binary sentiment (positive/negative), there would be 2 logits.
# For 3 classes (positive/negative/neutral), there would be 3 logits.
# By default, distilbert-base-uncased is not fine-tuned for classification,
# so it just has a default number of classes (usually 2, as it's often set up for binary tasks).
logits = outputs.logits
print(f"Raw Logits from model: {logits}") # e.g., tensor([[-0.2467,  0.1362]])

# The `hidden_states` contain the embeddings from all layers.
# The last hidden state is often used for downstream tasks.
# `hidden_states[0]` is the embedding layer output (input embeddings before any transformer layers).
# `hidden_states[-1]` is the output of the *last* Transformer layer (contextualized embeddings).
# The shape will be (batch_size, sequence_length, hidden_size).
# For `distilbert-base-uncased`, hidden_size is 768.
last_hidden_state_embeddings = outputs.hidden_states[-1]
print(f"Shape of Last Layer Contextual Embeddings: {last_hidden_state_embeddings.shape}")
# Example: torch.Size([1, 12, 768]) for our sample sentence.
# 1 (batch size), 12 (number of tokens), 768 (embedding dimension)

# To get the embedding for the [CLS] token (often used for classification):
cls_embedding = last_hidden_state_embeddings[:, 0, :]
print(f"Shape of [CLS] Token Embedding: {cls_embedding.shape}")
# Example: torch.Size([1, 768]) - this single vector represents the entire sentence's context
# and is passed to the classification head.

# --- Step 5: Convert Logits to Probabilities and Predict Sentiment ---
# We apply a softmax function to the logits to get probabilities across classes.
print("\n--- Step 5: Probabilities and Prediction ---")
probabilities = torch.softmax(logits, dim=1)
print(f"Probabilities (e.g., for 2 classes): {probabilities}")

# To get the predicted class index (0 or 1 for binary classification)
predicted_class_idx = torch.argmax(probabilities, dim=1).item()
print(f"Predicted Class Index: {predicted_class_idx}")

# Note: Without knowing the specific fine-tuning mapping (e.g., 0=negative, 1=positive),
# these indices are abstract for `model_base`.
# For real sentiment analysis, you'd use a model already fine-tuned with labels.

# --- Step 6: Using a Pre-trained Sentiment Analysis Pipeline (End-to-End) ---
# For practical sentiment analysis, you'd typically use a model already fine-tuned for the task.
# Hugging Face `pipeline` abstracts away much of the above.
print("\n--- Step 6: Using a Dedicated Sentiment Analysis Pipeline ---")
# This loads a model already fine-tuned for sentiment, complete with appropriate labels.
# E.g., 'nlptown/bert-base-multilingual-uncased-sentiment' is a common one for 5-star sentiment.
# Let's use a simpler one if available, or just demonstrate the concept.
# A common choice for English sentiment is 'distilbert-base-uncased-finetuned-sst-2-english'
# but it's better to use one that clearly outputs "positive" or "negative".
# For simplicity, let's use a popular readily available one.
try:
    sentiment_pipeline = pipeline("sentiment-analysis", model="finiteautomata/bertweet-base-sentiment-analysis")
    print(f"Model loaded: finiteautomata/bertweet-base-sentiment-analysis")

    result = sentiment_pipeline(text)
    print(f"Sentence: '{text}'")
    print(f"Sentiment Analysis Result: {result}")
    # Example Output: [{'label': 'POS', 'score': 0.9989}]

    text_negative = "This product broke after one day, completely useless."
    result_negative = sentiment_pipeline(text_negative)
    print(f"\nSentence: '{text_negative}'")
    print(f"Sentiment Analysis Result: {result_negative}")

except Exception as e:
    print(f"Could not load specific sentiment model: {e}")
    print("Falling back to a general example or skipping pipeline demo.")
    print("The key takeaway is that the fine-tuned model leverages the pre-trained embeddings.")


print("\n--- Sentiment Analysis Process Complete ---")
print("Key takeaway: Pre-trained embeddings provide meaningful numerical representations that a classification head then uses to predict sentiment.")

--- Step 1: Loading Tokenizer and Model ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/100 [00:00<00:00, 25731.93it/s, Materializing param=distilbert.embeddings.LayerNorm.bias]

Loading weights:   1%|          | 1/100 [00:00<00:00, 816.49it/s, Materializing param=distilbert.embeddings.LayerNorm.bias]  

Loading weights:   2%|▏         | 2/100 [00:00<00:00, 981.47it/s, Materializing param=distilbert.embeddings.LayerNorm.weight]

...


RobertaForSequenceClassification LOAD REPORT from: finiteautomata/bertweet-base-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Model loaded: finiteautomata/bertweet-base-sentiment-analysis


Sentence: 'This movie was absolutely fantastic! I loved every minute of it.'
Sentiment Analysis Result: [{'label': 'POS', 'score': 0.9918115139007568}]



Sentence: 'This product broke after one day, completely useless.'
Sentiment Analysis Result: [{'label': 'NEG', 'score': 0.9826966524124146}]

--- Sentiment Analysis Process Complete ---
Key takeaway: Pre-trained embeddings provide meaningful numerical representations that a classification head then uses to predict sentiment.
